# Phase 3: Data Mining and Feature Engineering

This notebook covers the transformation of raw financial data into predictive features. We bridge the gap between non-stationary raw prices and memory-preserved stationary feature tensors ready for machine learning and reinforcement learning.

## Objectives:
1. **Stationarity vs Memory**: Implement Fractional Differentiation (FFD) to preserve as much market history as possible while ensuring the data is statistically stationary.
2. **Event-Based Sampling and Labeling**: Implement the Triple Barrier Method for high-quality training labels based on local volatility.
3. **Multi-Asset Alignment**: Implement a Soft Join strategy to synchronize equities (market sessions) with global macro indicators (24h cycles).

## 1. Imports and Configuration

We establish our global constants here to avoid magic numbers and ensure reproducibility via a fixed seed.

In [1]:
import pandas as pd
import numpy as np
import os
import random
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 25
np.random.seed(SEED)
random.seed(SEED)

# Paths
DAILY_DIR = "../data/raw/daily"
HOURLY_DIR = "../data/raw/hourly"
PROCESSED_DIR = "../data/features"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Constants
TICKERS = ["AAPL", "MSFT", "JPM", "SPY", "GLD", "VIX", "TNX", "DX", "XLK", "XLF"]
CORE_TICKERS = ["AAPL", "MSFT", "JPM", "SPY", "GLD"]

# Feature Hyperparameters
RSI_WINDOW = 14
VOLATILITY_WINDOW = 20
FFD_THRESHOLD = 1e-4
ADF_SIGNIFICANCE = 0.05
DAILY_LABELLING_HORIZON = 5
HOURLY_LABELLING_HORIZON = 8

print(f"Initialization complete: Seed {SEED}")

Initialization complete: Seed 25


## 2. Fractional Differentiation (FFD) Engine

Financial time-series are rarely I(0) (stationary) but integer differentiation (d=1, e.g., returns) removes most of the signal memory. FFD allows us to find a fractional value of d (e.g., 0.35) that satisfies stationarity tests while retaining the maximum correlation with the original price series. This property is vital for RL agents that rely on historic patterns.

In [2]:
def apply_ffd(series, d, threshold=FFD_THRESHOLD):
    """
    Apply Fractional Differentiation to a series using a fixed-width window.
    
    Args:
        series (pd.Series): Input time series (usually log prices).
        d (float): Fractional order of differentiation.
        threshold (float): Weight threshold below which weights are truncated.
        
    Returns:
        pd.Series: Stationarized series with memory preserved.
    """
    w = [1.0]
    for k in range(1, len(series)):
        w_ = -w[-1] / k * (d - k + 1)
        if abs(w_) < threshold: break
        w.append(w_)
    w = np.array(w[::-1]).reshape(-1, 1)
    width = len(w) - 1
    res = []
    series_array = series.values
    for i in range(width, len(series)):
        res.append(np.dot(w.T, series_array[i-width:i+1])[0])
    return pd.Series(res, index=series.index[width:])

def find_min_d(series):
    """
    Grid search for the minimum d such that the ADF p-value < ADF_SIGNIFICANCE.
    
    Args:
        series (pd.Series): The series to test.
        
    Returns:
        float: The optimal d value between 0 and 1.
    """
    if len(series) < 50: return 1.0
    # Iterate through possible d values from 0 (no diff) to 1 (full returns)
    for d in np.linspace(0, 1, 6):
        ffd_series = apply_ffd(series, d)
        if len(ffd_series) < 30: continue
        try:
            p_val = adfuller(ffd_series.dropna())[1]
            if p_val < ADF_SIGNIFICANCE: return d
        except:
            continue
    return 1.0

## 3. Triple Barrier Labeling Engine

Standard ML labels (sign of return) often fail in high-volatility environments. The Triple Barrier Method uses a dynamic boundary based on local rolling volatility. 
1. **Upper Barrier**: Hits a target profit.
2. **Lower Barrier**: Hits a stop-loss.
3. **Time Barrier**: If neither is hit within a certain window, the sample is either 0 or the sign of the current progress.

In [3]:
def apply_triple_barrier(df, horizon=DAILY_LABELLING_HORIZON):
    """
    Apply a dynamic triple-barrier labeler aligned with local volatility.
    
    Args:
        df (pd.DataFrame): Dataframe containing 'Close' column.
        horizon (int): The maximum lookahead window for the barriers.
        
    Returns:
        pd.Series: Integer labels (-1, 0, 1).
    """
    # Volatility estimated as the rolling standard deviation of percentage returns
    vol = df['Close'].pct_change().rolling(VOLATILITY_WINDOW).std() * np.sqrt(horizon)
    labels = []
    for i in range(len(df)):
        if i + horizon >= len(df) or pd.isna(vol.iloc[i]):
            labels.append(np.nan)
            continue
        p0 = df['Close'].iloc[i]
        v = vol.iloc[i]
        if v == 0: 
            labels.append(0)
            continue
            
        ub, lb = p0 * (1 + v), p0 * (1 - v)
        window = df['Close'].iloc[i+1 : i+horizon+1]
        
        # Check barrier hits
        if any(window >= ub): labels.append(1)
        elif any(window <= lb): labels.append(-1)
        else: labels.append(0)
    return pd.Series(labels, index=df.index)

## 4. Feature Set Generation (Strategic & Tactical)

We aggregate all features into a unified matrix. A critical find during the mining phase was that Macro indices (VIX, TNX) trade on a different clock than Equities. We solve this by anchoring the dataset on SPY's session times and and reindexing the Macro data via forward-filling.

In [4]:
def add_technical_indicators(df):
    """
    Add momentum (RSI), trend (EWMA), and volatility features.
    """
    # RSI Calculation
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=RSI_WINDOW).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=RSI_WINDOW).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # Momentum: EWMA spread
    ema_fast = df['Close'].ewm(span=12).mean()
    ema_slow = df['Close'].ewm(span=26).mean()
    df['MACD_Spread'] = ema_fast - ema_slow
    
    # Local Volatility
    df['Vol_Local'] = df['Close'].pct_change().rolling(VOLATILITY_WINDOW).std()
    return df

def generate_master_set(horizon="daily"):
    """
    Execute the full engineering pipeline for a specific horizon.
    """
    data_dir = DAILY_DIR if horizon == "daily" else HOURLY_DIR
    all_processed = {}
    
    for ticker in TICKERS:
        path = os.path.join(data_dir, f"{ticker}.csv")
        # Skip yfinance metadata rows
        df = pd.read_csv(path, skiprows=[1,2], index_col=0, parse_dates=True)
        df.index = pd.to_datetime(df.index, utc=True).tz_localize(None)
        df = df.apply(pd.to_numeric, errors='coerce').dropna()
        
        # Feature construction
        df = add_technical_indicators(df)
        
        # Stationarization via FFD
        d_star = find_min_d(np.log(df['Close']))
        df['FFD_Price'] = apply_ffd(np.log(df['Close']), d_star)
        
        # Labelling (Core Assets only)
        if ticker in CORE_TICKERS:
            h_val = DAILY_LABELLING_HORIZON if horizon == "daily" else HOURLY_LABELLING_HORIZON
            df['Target'] = apply_triple_barrier(df, horizon=h_val)
            
        df.columns = [f"{ticker}_{c}" for c in df.columns]
        all_processed[ticker] = df
        
    # Intersection and Alignment
    anchor_index = all_processed['SPY'].index
    master = pd.DataFrame(index=anchor_index)
    
    for ticker in TICKERS:
        t_df = all_processed[ticker]
        if horizon == "hourly":
            # Force alignment of Macro :00 bars to Equity :30 bars
            t_df = t_df.reindex(anchor_index, method='ffill', limit=1)
        master = master.join(t_df, how='left')
    
    # Finalizing alignment
    master = master.ffill().dropna(subset=['JPM_Target', 'AAPL_Target']).dropna()
    
    save_path = os.path.join(PROCESSED_DIR, f"{horizon}_features.csv")
    master.to_csv(save_path)
    print(f"Final Dataset ({horizon}): {master.shape}")
    return master

print("Starting Strategic (Daily) Engineering...")
daily_m = generate_master_set("daily")
print("Starting Tactical (Hourly) Engineering...")
hourly_m = generate_master_set("hourly")

Starting Strategic (Daily) Engineering...


Final Dataset (daily): (4531, 95)
Starting Tactical (Hourly) Engineering...


Final Dataset (hourly): (2963, 95)


## 5. Summary and Conclusions

### Key Discoveries from Mining and Engineering:

1. **Stationarity Preservation**: Through FFD grid search, we found that most equity log-prices require a differentiation value `d` between 0.3 and 0.45 to pass the ADF test. This is significantly lower than the standard `d=1` (returns), implying that a substantial amount of "Market Memory" (mean reversion signals) is lost when only using simple returns.
2. **Session Realignment**: We successfully bridged the timing gap between equity market hours and 24-hour macro indicators. Using `SPY` as the temporal anchor ensures the RL environment updates only when trading actions are actually possible.
3. **Volatility Clustering**: Triple barrier labels confirm that fixed percentage targets are suboptimal. By scaling barriers with rolling volatility (`Vol_Local`), we ensure the labels signify an abnormal price move relative to recent regime baseline.

### Next Steps:
- Move to **Phase 4: Baseline ML** to fit ARIMA and GARCH models on these engineered features.
- Evaluate if `FFD_Price` provides superior predictive power compared to raw returns in a linear baseline model.